# Stickman Studio · Voice server

Turns your script into a Hindi / Hinglish / English voiceover with **Chatterbox Multilingual** (free, MIT licence) on Colab's free GPU, and can clone **your own voice** from a short recording.

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run cell **1** (installs, about 3–5 minutes). The runtime restarts by itself once; that is expected.
3. Run cell **2**. The first time it downloads the voice model and the speech recogniser used to clean your samples (about 3.5 GB together). It then prints a link like `https://something.trycloudflare.com`.
4. On your phone, tap **Open in Stickman Studio** under cell 2, then **Connect** in the app. (Or copy the link and paste it in Stickman Studio → **Script + Voice → AI voice** → **Connect**.) Keep this tab open while you make voiceovers.

Colab's free GPU used up for today ("No GPU" or "Cannot connect to GPU backend")? Use the **Kaggle notebook** instead: https://www.kaggle.com/kernels/welcome?src=https://github.com/gopalkrishan680-hash/GK/blob/claude/stickman-video-voiceover-sync-ivss4s/stickman-voice/StickmanVoiceKaggle.ipynb

The link changes every time you start the server. Colab's free GPU stops after a few hours or when the tab is idle for long; then just run cell 2 again.

Only clone your own voice, or a voice whose owner has given you permission. Audio made by Chatterbox carries Resemble AI's inaudible watermark.

In [ ]:
#@title 1 · Install (once per session; restarts the runtime)
import os, subprocess, sys
print('Stickman voice notebook v8')
if os.path.exists('/content/.ss_voice_installed_v4'):
    print('Already installed. Run cell 2.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chatterbox-tts==0.1.7', 'fastapi', 'uvicorn', 'noisereduce', 'faster-whisper'], check=True)
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    # Chatterbox pins PyTorch 2.6; Colab's torchvision is built for a newer PyTorch and breaks
    # `transformers` ("torchvision::nms does not exist"). Install the build that matches PyTorch 2.6.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchvision==0.21.0', '--no-deps'])
    os.chmod('/content/cloudflared', 0o755)
    open('/content/.ss_voice_installed_v4', 'w').write('ok')
    print('Installed. The runtime restarts now (this is expected). Then run cell 2.')
    os.kill(os.getpid(), 9)

In [ ]:
#@title 2 · Start the voice server and get the link
BRANCH = "claude/stickman-video-voiceover-sync-ivss4s" #@param {type:"string"}
MODEL = "hindi" #@param ["hindi", "v3", "v2"]
HF_TOKEN = "" #@param {type:"string"}
import os, re, subprocess, sys, threading, time, urllib.request, importlib.util
print('Stickman voice notebook v8')
def show_link(url):
    '''Big button that opens the Stickman Studio app and offers to connect, plus the link to copy.'''
    from IPython.display import HTML, display
    import urllib.parse
    fallback = 'https://github.com/gopalkrishan680-hash/GK/releases/tag/android-latest'
    app = ('intent://connect?url=' + urllib.parse.quote(url, safe='') +
           '#Intent;scheme=stickmanstudio;package=com.gopalkrishan.stickmanstudio;S.browser_fallback_url=' +
           urllib.parse.quote(fallback, safe='') + ';end')
    display(HTML(f'''<div style="font:16px sans-serif;padding:14px;border:3px solid #f5c518;border-radius:10px">
      <b>Voice server is running.</b><br>
      <a href="{app}" target="_blank" style="display:inline-block;margin:12px 0;padding:14px 20px;background:#f5c518;color:#000;border-radius:10px;font-weight:bold;font-size:18px;text-decoration:none">&#128241; Open in Stickman Studio</a>
      <div>On a phone with the app: tap the button, then tap <b>Connect</b> in the app.<br>
      Otherwise copy this link (the app notices it when you switch back to it), or paste it in Stickman Studio &rarr; AI voice &rarr; Connect:</div>
      <input value="{url}" readonly onclick="this.select()" style="width:100%;font:18px monospace;margin-top:8px;padding:6px;box-sizing:border-box">
      <button onclick="navigator.clipboard.writeText('{url}').then(()=>this.textContent='Copied')" style="margin-top:8px;padding:8px 14px;font-size:16px">Copy link</button>
      <div style="margin-top:6px;color:#666">Keep this tab open while you make voiceovers.</div></div>'''))
    print(url)
if not os.path.exists('/content/.ss_voice_installed_v4'):
    raise SystemExit('Run cell 1 first (the install changed in notebook v4). It restarts the runtime once; then run this cell.')
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Menu Runtime → Change runtime type → T4 GPU, then run cell 1 and 2 again. '
                     'If T4 cannot be chosen, Colab\'s free GPU is used up for today: use the Kaggle notebook (link at the top).')

# A server from an earlier run of this cell keeps the old code and the GPU memory: start clean.
if os.path.exists('/content/.ss_server_pid') and open('/content/.ss_server_pid').read() == str(os.getpid()):
    print('A server from an earlier run is still active. Restarting the runtime so the new version loads; then run cell 2 again.')
    os.remove('/content/.ss_server_pid')
    os.kill(os.getpid(), 9)

# Download the server code of the newest commit (raw links of a branch name are cached for minutes).
import json
try:
    ref = json.load(urllib.request.urlopen(f'https://api.github.com/repos/gopalkrishan680-hash/GK/commits/{BRANCH}', timeout=20))['sha']
except Exception:
    ref = BRANCH
urllib.request.urlretrieve(f'https://raw.githubusercontent.com/gopalkrishan680-hash/GK/{ref}/stickman-voice/server.py?t={int(time.time())}', '/content/ss_server.py')
sys.path.insert(0, '/content')
import importlib, ss_server
importlib.reload(ss_server)

print('Server code version', ss_server.SERVER_VERSION, '· commit', ref[:7])
os.environ['STICKMAN_MODEL'] = MODEL           # hindi = Resemble AI's Hindi pack (falls back to V3, then V2)
if not HF_TOKEN.strip():   # empty box: use the token saved in Colab Secrets (key icon on the left), if any
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
        if HF_TOKEN:
            print('Using HF_TOKEN from Colab Secrets')
    except Exception:
        HF_TOKEN = ''
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()   # only needed if the Hindi pack asks you to accept its terms
print('Loading the voice model (first time: downloading a few GB)…')
engine = ss_server.start(port=8000)
open('/content/.ss_server_pid', 'w').write(str(os.getpid()))
print('Model ready on', engine.device, '· model:', getattr(engine, 'variant', '?'))
if getattr(engine, 'notes', None):
    print('Note:', ' | '.join(engine.notes))

tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    if time.time() - t0 > 90:
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep draining the log

if not url:
    print('Could not open the public link. Run this cell again.')
else:
    show_link(url)

### Troubleshooting
- **"No GPU"**: Runtime → Change runtime type → T4 GPU, then run cells 1 and 2 again. If T4 cannot be chosen or Colab cannot connect to a GPU, today's free GPU time is used up: use the Kaggle notebook (link at the top) or try again tomorrow.
- **Stickman Studio says it cannot reach the server**: the link changes on every start; copy the new one. Also check this tab is still running.
- **A line sounds wrong** (skipped or extra words): in Stickman Studio press **Redo** on that line; it tries again with a new random seed.
- **My cleaned sample sounds cut too much**: in Stickman Studio untick *Clean my samples first* (or *Make it crystal clear*) and compare with **Original**.
- **The voice sounds robotic**: record a new sample in which you *narrate with feeling* at your normal pace (15–30 s), choose *Speaking pace: Relaxed*, and keep MODEL = hindi. Cell 2 prints which model was loaded; if it says the Hindi pack was skipped because of its terms, open huggingface.co/ResembleAI/Chatterbox-Multilingual-hi, accept them, create a token (Settings → Access Tokens) and paste it into HF_TOKEN.
- **Hindi sounds English**: write Hindi in Devanagari (हिंदी). English words can stay in English.